In [1]:
from dataclasses import dataclass
from typing import List
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
import json
import time

# Data Schema
@dataclass
class Message:
    message_id: str
    author: str
    date_raw: str
    body: str
    url: str

@dataclass
class Thread:
    thread_id: str
    subject: str
    messages: List[Message]

# Retriever
class Retriever:
    def __init__(self, json_path: str):
        with open(json_path, "r") as f:
            self.threads = json.load(f)
        self.docs=[]

    def load_documents(self, max_chars: int = 400) -> List[str]:
        documents = []
        for thread in self.threads:
            subject = thread.get("subject", "")
            for msg in thread.get("messages", []):
                body = msg.get("body", "")
                preview = (body[:max_chars] + "...") if len(body) > max_chars else body
                documents.append(f"Thread: {subject}\nBody: {preview}")
        self.docs = documents
        return documents

# Query Understanding
class QueryUnderstanding:
    def __init__(self, model_name: str = "facebook/bart-large-mnli"):
        self.classifier = pipeline("zero-shot-classification", model=model_name)

    def normalize(self, query: str) -> str:
        query = query.strip()
        query = query.replace("\n", " ").lower()
        return query

    def classify(self, query: str) -> dict:
        candidate_labels = [
            "Technical Question",
            "Concept Explanation",
            "Error Troubleshooting",
            "Software Usage",
            "General Discussion",
        ]
        result = self.classifier(query, candidate_labels)
        return {
            "query": query,
            "top_label": result["labels"][0],
            "scores": dict(zip(result["labels"], result["scores"])),
        }

    def process(self, query: str) -> dict:
        normalized = self.normalize(query)
        classification = self.classify(normalized)
        return classification

# Semantic Filter
class SemanticFilter:
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def filter(self, query: str, documents: List[str], top_k: int = 5) -> List[str]:
        doc_embeddings = self.model.encode(documents, convert_to_tensor=True)
        query_embedding = self.model.encode(query, convert_to_tensor=True)
        hits = util.semantic_search(query_embedding, doc_embeddings, top_k=top_k)[0]
        return [documents[h['corpus_id']] for h in hits]

# Prompt Builder
def build_prompt(query: str, retrieved_docs: List[str], max_chars: int = 2000) -> str:
    context = "\n".join(retrieved_docs)
    if len(context) > max_chars:
        context = context[:max_chars] + "..."
    return f"Context:\n{context}\n\nQuestion: {query}"

# LLM Integration
class HuggingFaceLLM:
    def __init__(self, model_name="google/flan-t5-large"):
        self.generator = pipeline("text2text-generation", model=model_name)

    def generate(self, prompt: str) -> str:
        result = self.generator(prompt, max_new_tokens=100)
        return result[0]["generated_text"]

# Demo Workflow
if __name__ == "__main__":
    
    retriever = Retriever("thread_level.json")
    documents = retriever.load_documents()[:10]

    query = "Is there a way to define a repulsive potential between the two proteins without specifying a specific pulling direction?"
    query_understanding = QueryUnderstanding()
    query_info = query_understanding.process(query)

    print("\n=== Query Understanding ===")
    print(f"Normalized Query: {query_info['query']}")
    print(f"Detected Intent: {query_info['top_label']}")
    print("Scores:", query_info["scores"])
    
    semantic_filter = SemanticFilter("sentence-transformers/all-MiniLM-L6-v2")

    start = time.perf_counter()
    doc_embeddings = semantic_filter.model.encode(documents, convert_to_tensor=True)
    end = time.perf_counter()
    print(f"\nDocument embedding time: {end - start:.4f} seconds for {len(documents)} docs")
    
    start = time.perf_counter()
    query_embedding = semantic_filter.model.encode(query, convert_to_tensor=True)
    hits = util.semantic_search(query_embedding, doc_embeddings, top_k=3)[0]
    end = time.perf_counter()
    print(f"Query retrieval time: {end - start:.4f} seconds")

    retrieved_docs = [documents[h['corpus_id']] for h in hits]
    
    print("\nRetrieved docs:")
    for doc in retrieved_docs:
        print("-", doc, "\n")

    prompt = build_prompt(query, retrieved_docs)
    print("\nPrompt sent to LLM:\n", prompt)

    llm = HuggingFaceLLM()
    answer = llm.generate(prompt)

    print("\nAnswer:", answer)

Device set to use mps:0



=== Query Understanding ===
Normalized Query: is there a way to define a repulsive potential between the two proteins without specifying a specific pulling direction?
Detected Intent: Technical Question
Scores: {'Technical Question': 0.5998063683509827, 'Concept Explanation': 0.19144023954868317, 'General Discussion': 0.11070568114519119, 'Error Troubleshooting': 0.0560557097196579, 'Software Usage': 0.04199196770787239}

Document embedding time: 1.2250 seconds for 10 docs
Query retrieval time: 0.8559 seconds

Retrieved docs:
- Thread: [AMBER] Metadynamics question
Body: Dear AMBER community,
I’d like to simulate the unbinding of a protein from a receptor complex. My initial thought was to use steered MD, but is there a way in AMBER to instead define a repulsive potential between the two proteins without specifying a specific pulling direction?
Best,
Matthew 

- Thread: [AMBER] Metadynamics question
Body: You can use a distance restraint with  r2 farther than current distance.
Maybe r

Device set to use mps:0



Answer: section 29.1 of the Amber 22 manual gives details... basically I would set up a COM distance restraint, using whatever set of atoms for each group that you feel makes sense (maybe all or a subset of CA atoms of each protein?). then, look at the R1 and R2 values, R2 will specify the location of the minimum. You could either set R2 to the current distance between the groups (measured using c
